<a href="https://colab.research.google.com/github/rwcitek/Medicare-data/blob/data/medicare_catalog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import duckdb
import pandas as pd


In [ ]:
states = [
'State=AK',
'State=AL',
'State=AR',
'State=AS',
'State=AZ',
'State=CA',
'State=CO',
'State=CT',
'State=DC',
'State=DE',
'State=FL',
'State=GA',
'State=GU',
'State=HI',
'State=IA',
'State=ID',
'State=IL',
'State=IN',
'State=KS',
'State=KY',
'State=LA',
'State=MA',
'State=MD',
'State=ME',
'State=MH',
'State=MI',
'State=MN',
'State=MO',
'State=MP',
'State=MS',
'State=MT',
'State=NC',
'State=ND',
'State=NE',
'State=NH',
'State=NJ',
'State=NM',
'State=NV',
'State=NY',
'State=OH',
'State=OK',
'State=OR',
'State=PA',
'State=PR',
'State=RI',
'State=SC',
'State=SD',
'State=TN',
'State=TX',
'State=UT',
'State=VA',
'State=VI',
'State=VT',
'State=WA',
'State=WI',
'State=WV',
'State=WY',
]
states

In [ ]:
base = "https://github.com/rwcitek/Medicare-data/raw/refs/heads/data/medicare"
parquet = "381a603c494b4599ba466d17487a67ab-0.parquet"
urls = [ f"{base}/{s}/{parquet}" for s in states ]
urls


## Test querying the hive parquets

In [ ]:
with duckdb.connect() as con:
  con.execute("INSTALL httpfs; LOAD httpfs;")
  df = con.query(f'''
    select count(1) as provider_count
    FROM read_parquet(
      {urls},
      hive_partitioning = true
    )
  ''').df()
df


## Create the catalog file


In [ ]:
# 1. Connect to a persistent file (this creates 'medicare_catalog.db')
with duckdb.connect('medicare_catalog.db') as con:
  # 2. Load the necessary extensions
  con.execute("INSTALL httpfs; LOAD httpfs;")

  # 3. Create the View using the list of URLs
  # We use an f-string or the list directly to define the view
  con.execute(f'''
    CREATE OR REPLACE VIEW remote_medicare AS
      SELECT *
      FROM read_parquet(
        {urls},
        hive_partitioning = true
      )
  ''')


## Use the catalog file after uploading to GitHub


In [ ]:
remote_url = "https://github.com/rwcitek/Medicare-data/raw/refs/heads/data/medicare_catalog.db"

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"ATTACH '{remote_url}' AS remote_medicare (READ_ONLY)")
con.execute(f'''CREATE OR REPLACE VIEW medicare AS SELECT * FROM remote_medicare.main.remote_medicare ''')


In [ ]:
df = con.execute("SELECT * FROM medicare LIMIT 5").df()
df


In [ ]:
count = con.execute("SELECT count(1) FROM medicare").df().iloc[0,0]
f"{count:,}"


In [ ]:
# Returns a tuple like (123456,), so we grab index 0
count = con.execute("SELECT count(1) FROM medicare").fetchone()[0]
print(f"{count:,}")


In [ ]:
# # This is increadibly slow: 345s vs 14s

# count = con.table("medicare").count("*").fetchone()[0]
# print(f"{count:,}")
